# Final Notebook

## Analyst Decisions

- Creating a Pivot Chart for each Brench containing the number of events for each catagory:
    1. Children: 0 to 12
    2. Teens : 13 to 17
    3. Adults : 18+

***Note: Might need to reclassify the Adults into three categories: Young Adults, Adults, and Seniors.***

- Then, making a conclusion page that will include:
   - Top 15 Branches
   - Lowest 15 branches
   - Librarys that have more space/ computer hubs (Resources)
   - Regestiration to the library: how many children/Adults are registered



In [1]:
# Import Lib
# %matplotlib notebook
import pandas as pd
import sweetviz as sv
import matplotlib.pyplot as plt #this is for Graph creation.
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import interact
import mplcursors


# Explore DF function
def explore (Data):
    print(Data.head(5))
    print(Data.info())
    M_Values = Data.isnull().sum().sort_values(ascending=False) 
    print(M_Values)
    print(Data.duplicated().sum())
    missing = M_Values.index[0]
    print(Data[Data[missing].isnull()])

# DataFrame creation

tpl_BGI_2023 = pd.read_csv("data/tpl-branch-general-information-2023.csv",index_col=0)
tpl_BSR_2024 = pd.read_csv("data/tpl-branch-space-rentals-2024.csv",index_col=0)
tpl_RABB = pd.read_csv("data/tpl-card-registrations-annual-by-branch.csv",index_col=0)
tpl_EFS2 = pd.read_csv("data/tpl-events-feed_source2.csv",index_col=0)
tpl_VABB = pd.read_csv("data/tpl-visits-annual-by-branch.csv",index_col=0)
tpl_WUABB_2018_2023 = pd.read_csv("data/tpl-workstation-usage-annual-by-branch-2018-2023.csv",index_col=0)
tpl_LCBCT = pd.read_csv("data/library-circulation-by-cardholder-type.csv",index_col=0)

tpl_BGI_2023.head(5)

,BranchCode,PhysicalBranch,BranchName,Address,PostalCode,Website,Telephone,SquareFootage,PublicParking,KidsStop,...,Workstations,ServiceTier,Lat,Long,NBHDNo,NBHDName,TPLNIA,WardNo,WardName,PresentSiteYear
_id,,,,,,,,,,,,,,,,,,,,,
1,AB,1,Albion,"1515 Albion Road, Toronto, ON, M9V 1B2",M9V 1B2,https://www.tpl.ca/albion,416-394-5170,29000,59,1.0,...,38.0,DL,43.739826,-79.584096,2.0,Mount Olive-Silverstone-Jamestown,1.0,1.0,Etobicoke North,2017.0
2,ACD,1,Albert Campbell,"496 Birchmount Road, Toronto, ON, M1K 1N8",M1K 1N8,https://www.tpl.ca/albertcampbell,416-396-8890,28957,45,0.0,...,36.0,DL,43.708019,-79.269252,120.0,Clairlea-Birchmount,1.0,20.0,Scarborough Southwest,1971.0
3,AD,1,Alderwood,"2 Orianna Drive, Toronto, ON, M8W 4Y1",M8W 4Y1,https://www.tpl.ca/alderwood,416-394-5310,7341,shared,0.0,...,7.0,NL,43.601944,-79.547252,20.0,Alderwood,0.0,3.0,Etobicoke-Lakeshore,1999.0
4,AG,1,Agincourt,"155 Bonis Avenue, Toronto, ON, M1T 3W6",M1T 3W6,https://www.tpl.ca/agincourt,416-396-8943,27000,86,0.0,...,42.0,DL,43.785167,-79.293430,118.0,Tam O'Shanter-Sullivan,0.0,22.0,Scarborough-Agincourt,1991.0
5,AH,1,Armour Heights,"2140 Avenue Road, Toronto, ON, M5M 4M7",M5M 4M7,https://www.tpl.ca/armourheights,416-395-5430,2988,shared,0.0,...,5.0,NL,43.739337,-79.421889,39.0,Bedford Park-Nortown,0.0,8.0,Eglinton-Lawrence,1982.0


## Cleaning Decisions:

#### For table ***tpl-branch-general-information-2023*** 
1. Clumns removed: 
    - Location:  
    'Address',
    'PostalCode',
    'Lat',
    'Long',
    'WardName',
    'NBHDNo',
    'WardNo',
    - Program Indicators: 
    'AdultLiteracyProgram',
    'LeadingReading',
    - Operational: 
    'SquareFootage',
    'PublicParking',
    'Website',
    'Telephone',
    'ServiceTier',
    'PresentSiteYear'

2. Raws that contain 'KidsStop' were kept, (might need to change)
3. Brench name: 'Jane/Dundas', was changed to its current name: 'Daniel G. Hill'
4. The new DF called: tpl_BGI_2023_cl 



In [2]:
def clean_branches(df):
    df =  df[df['KidsStop'].notnull()]
    drop_columns = [
    # Location
    'Address',
    'PostalCode',
    'Lat',
    'Long',
    'WardName',
    'NBHDNo',
    'WardNo',
    # Program Indicators
    'AdultLiteracyProgram',
    'LeadingReading',
    # Operational
    'SquareFootage',
    'PublicParking',
    'Website',
    'Telephone',
    'ServiceTier',
    'PresentSiteYear']
    df = df.drop(columns = drop_columns)
    return df

tpl_BGI_2023_cl = clean_branches(tpl_BGI_2023)

tpl_BGI_2023_cl['BranchName'] = tpl_BGI_2023_cl['BranchName'].str.replace(
    'Jane/Dundas', 'Daniel G. Hill'
)


tpl_BGI_2023_cl['BranchName'] = tpl_BGI_2023_cl['BranchName'].str.replace(
    'Perth/Dupont', 'Junction Triangle'
)

# branch_names = set(tpl_BGI_2023_cl['BranchName'].unique())
branch_names = set(tpl_BGI_2023_cl['BranchName'].astype(str).str.strip().str.lower())
# len(tpl_BGI_2023_cl['BranchName'].unique())
print(branch_names)
print(len(branch_names))

{'deer park', 'parliament street', 'bendale', 'annette street', 'evelyn gregory', 'highland creek', 'jane/sheppard', 'humber summit', "ethennonnhawahstihnen'", 'burrows hall', 'eatonville', 'steeles', 'malvern', 'amesbury park', 'queen/saulter', 'barbara frum', 'cedarbrae', 'college/shaw', 'north york central library', 'port union', 'sanderson', 'maryvale', 'albert campbell', 'flemingdon park', 'morningside', 'hillcrest', 'st. lawrence', 'yorkville', 'mount pleasant', 'humber bay', 'forest hill', 'york woods', 'kennedy/eglinton', 'runnymede', 'elmbrook park', 'st. james town', 'bloor/gladstone', 'mount dennis', 'riverdale', 'taylor memorial', 'fort york', 'cliffcrest', 'centennial', 'weston', 'lillian h. smith', 'st. clair/silverthorn', 'mimico centennial', 's. walter stewart', 'city hall', 'dawes road', 'spadina road', 'beaches', 'danforth/coxwell', 'long branch', 'wychwood', 'don mills', 'dufferin/st. clair', 'alderwood', 'brookbanks', 'davenport', 'eglinton square', 'palmerston', 'v

#### Investigation Notes 
- Perth/Dupont, was closed in 2025 and moved to a new location called 'Junction Triangle'

- Now Junction Triangle is at 305 Campbell

- Jane/ Dundas Name was changed to Daniel G. Hill

- Libraries that don't show up on the events table:
    - todmorden room, humber bay, dawes road, pleasant view (Temp Closed),flemingdon park (Temp Closed), centennial (Closed for 3 years), 


#### For table ***tpl-events-feed_source2*** 

1. Columns removed:
        'EventID',
        'StartTime',
        'EndTime',
        'StartDateLocal',
        'RegistrationClosed',
        'FeaturedImageUrl'

2. Have 96 Locations, when Nan is Online events

In [3]:
# Table tpl-events-feed_source2 clean 

# print(tpl_EFS2['LastUpdatedOn'].unique())

def clean_branches(df):
    # remove all non physical branches
    # df = df[df[''].notnull()]
    drop_columns = [
        'EventID',
        'StartTime',
        'EndTime',
        'StartDateLocal',
        'RegistrationClosed',
        'FeaturedImageUrl'
    ]
    df = df.drop(columns = drop_columns)
    # df = df[~df['LocationName'].str.contains('Junction', case=True)]
    return df

tpl_EFS2_cl = clean_branches(tpl_EFS2)
# print(tpl_EFS2_cl.head(5))
# print(tpl_EFS2_cl.info())


tpl_EFS2_cl['LocationName'] = tpl_EFS2_cl['LocationName'].str.replace("Dufferin/St.Clair", "Dufferin/St. Clair")

tpl_EFS2_cl[tpl_EFS2_cl['LocationName'].isnull()]
events_locations = set(tpl_EFS2_cl['LocationName'].astype(str).str.strip().str.lower())
events_locations
branch_names = {x.lower() for x in branch_names}
diff_a = branch_names - events_locations
print(diff_a)

{'todmorden room', 'pleasant view', 'centennial', 'dawes road', 'flemingdon park', 'humber bay'}


## Merging the events with the breanch indo table


In [4]:
merged_data = tpl_BGI_2023_cl.merge(
    tpl_EFS2_cl,
    left_on='BranchName',      # column name in left table
    right_on='LocationName',     # column name in right table
    how='right'           # join type
)

merged_data.info()
merged_data.head(5)
# merged_data.Audiences.unique() 

<class 'pandas.DataFrame'>
RangeIndex: 8512 entries, 0 to 8511
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   BranchCode          8443 non-null   str    
 1   PhysicalBranch      8443 non-null   float64
 2   BranchName          8443 non-null   str    
 3   KidsStop            8443 non-null   float64
 4   CLC                 8443 non-null   float64
 5   DIH                 8443 non-null   float64
 6   TeenCouncil         8443 non-null   float64
 7   YouthHub            8443 non-null   float64
 8   Workstations        8443 non-null   float64
 9   NBHDName            8443 non-null   str    
 10  TPLNIA              8443 non-null   float64
 11  Title               8512 non-null   str    
 12  LocationName        8449 non-null   str    
 13  Audiences           8512 non-null   str    
 14  Languages           8512 non-null   str    
 15  EventTypes          8512 non-null   str    
 16  IsRecurring      

,BranchCode,PhysicalBranch,BranchName,KidsStop,CLC,DIH,TeenCouncil,YouthHub,Workstations,NBHDName,...,Title,LocationName,Audiences,Languages,EventTypes,IsRecurring,IsFull,Status,RegistrationIsFull,LastUpdatedOn
0,CL,1.0,North York Central Library,1.0,1.0,1.0,1.0,1.0,98.0,Willowdale West,...,Tea & Entertainment,North York Central Library,"Adults (18+), Older Adults",English,"Culture, Arts & Entertainment",False,False,ACTIVE,False,2026-02-26T12:22:14
1,CL,1.0,North York Central Library,1.0,1.0,1.0,1.0,1.0,98.0,Willowdale West,...,Tea & Entertainment,North York Central Library,"Adults (18+), Older Adults",English,"Culture, Arts & Entertainment",False,False,ACTIVE,False,2026-02-26T12:22:14
2,SWS,1.0,S. Walter Stewart,1.0,1.0,0.0,0.0,1.0,39.0,Danforth Village - East York,...,D&D for Teens: On-going Campaign,S. Walter Stewart,Teens (13-17),English,"Crafts & Hobbies, Games & Sports",True,False,ACTIVE,False,2026-02-26T12:22:14
3,SWS,1.0,S. Walter Stewart,1.0,1.0,0.0,0.0,1.0,39.0,Danforth Village - East York,...,S. Walter Stewart Seniors Social,S. Walter Stewart,Older Adults,English,"Crafts & Hobbies, Games & Sports",True,False,ACTIVE,False,2026-02-26T12:22:14
4,PL,1.0,Parliament Street,0.0,0.0,0.0,0.0,1.0,12.0,Moss Park,...,Finding Recovery Through Exercise Skills and H...,Parliament Street,Adults (18+),English,Personal Development,True,False,ACTIVE,False,2026-02-26T12:22:14


- The merged table include the events and the branch information columes that were not droped.

- The next step it to create a Graph and clasifiy the events Audiances catagories as followes:
    - IsChildren : Matches both 'Preschool Children' and 'School Age Children'
    - IsTeens: Matches 'Teens (13-17)'
    - IsAdults: Matches 'Adults (18+)' explicitly, avoiding 'Older Adults' and 'Younger Adults'
    - IsSeniors: Seniors / Older Adults (Matches 'Older Adults')
    - IsYoungerAdults: Matches 'Younger Adults (18-24)'

In [5]:
merged_data.Audiences.unique() 

<StringArray>
[                                                                                             'Adults (18+), Older Adults',
                                                                                                           'Teens (13-17)',
                                                                                                            'Older Adults',
                                                                                                            'Adults (18+)',
                                                                                                'Preschool Children (0-5)',
                                                                                              'School Age Children (6-12)',
                                                                      'Adults (18+), Older Adults, Younger Adults (18-24)',
                                                                    'Preschool Children (0-5), School Age Children (6-

In [6]:
# Step 1: Creating columns

# 1. Children (Matches both 'Preschool Children' and 'School Age Children')
merged_data['IsChildren'] = merged_data['Audiences'].str.contains('Children', na=False)

# 2. Teens (Matches 'Teens (13-17)')
merged_data['IsTeens'] = merged_data['Audiences'].str.contains('Teens', na=False)

# 3. Seniors / Older Adults (Matches 'Older Adults')
merged_data['IsSeniors'] = merged_data['Audiences'].str.contains('Older Adults', na=False)

# 4. Adults (Matches 'Adults (18+)' explicitly, avoiding 'Older Adults' and 'Younger Adults')
merged_data['IsAdults'] = merged_data['Audiences'].str.contains(r'Adults \(18\+\)', na=False)

# OPTIONAL: If you want to track 'Younger Adults (18-24)' separately
merged_data['IsYoungerAdults'] = merged_data['Audiences'].str.contains('Younger Adults', na=False)

# Step 2: Group by brench and sum:

events_by_group = merged_data.groupby('BranchName')[
    ['IsChildren', 'IsTeens', 'IsAdults', 'IsYoungerAdults', 'IsSeniors']
].sum()



- There are still the Nan location events, those are the online events.
I wii add this to the merged table and create the final version of the events catagorized in each branch to the rlvent group age.

In [7]:
# Online events - all NaN location rows from events table
online_events = tpl_EFS2_cl[tpl_EFS2_cl['LocationName'].isnull()].copy()
online_events['LocationName'] = 'Online'

online_events['IsChildren'] = online_events['Audiences'].str.contains('Children', na=False)
online_events['IsTeens'] = online_events['Audiences'].str.contains('Teens', na=False)
online_events['IsSeniors'] = online_events['Audiences'].str.contains('Older Adults', na=False)
online_events['IsAdults'] = online_events['Audiences'].str.contains(r'Adults \(18\+\)', na=False)
online_events['IsYoungerAdults'] = online_events['Audiences'].str.contains('Younger Adults', na=False)

events_by_group_online = online_events.groupby('LocationName')[
    ['IsChildren', 'IsTeens', 'IsAdults', 'IsYoungerAdults','IsSeniors']
].sum()

events_by_group_online = events_by_group_online.rename_axis('BranchName')

events_by_group
events_by_group_online

# EBB - Events By Brench
EBB_final = pd.concat([events_by_group, events_by_group_online])

EBB_final

,IsChildren,IsTeens,IsAdults,IsYoungerAdults,IsSeniors
BranchName,,,,,
Agincourt,40,21,39,3,25
Albert Campbell,53,11,59,4,21
Albion,60,26,19,1,0
Alderwood,20,0,2,0,1
Amesbury Park,67,3,32,2,20
...,...,...,...,...,...
Woodview Park,118,4,35,0,65
Wychwood,34,1,9,1,8
York Woods,69,40,76,4,62


In [8]:
arr = EBB_final.index.unique()
len(arr)

95

<!-- Graph Creation from here -->
## Graph Creations

1. Graph for each brench and its events catagorizes by group age

In [9]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import mplcursors


branch_selector = widgets.Dropdown(
    options=EBB_final.index.tolist(),
    description='Branch:'
)


def plot_branch(branch):
    data = EBB_final.loc[branch]
    data.plot(kind='bar', color=['#185FA5', '#854F0B', "#0F6E56", "#534AB7", "#993C1D"])
    plt.title(f'Events by age group — {branch}')
    plt.ylabel('Number of events')
    plt.xticks(rotation=0)
    plt.show()

# def plot_branch(branch):
#     plt.figure(figsize=(8, 5))

#     data = EBB_final.loc[branch]
#     ax = data.plot(        # ← change 1: save as ax
#         kind='bar',
#         color=['#185FA5', '#854F0B', "#0F6E56", "#534AB7", "#993C1D"],
#         # ax=plt.gca()
#     )
#     # data.plot(kind='bar', color=['#185FA5', '#854F0B', "#0F6E56", "#534AB7", "#993C1D"])
#     plt.title(f'Events by age group — {branch}')
#     plt.ylabel('Number of events')
#     plt.xticks(rotation=0)

#     mplcursors.cursor(ax, hover=True) 
#     plt.show()

widgets.interact(plot_branch, branch=branch_selector)

interactive(children=(Dropdown(description='Branch:', options=('Agincourt', 'Albert Campbell', 'Albion', 'Alde…

<function __main__.plot_branch(branch)>

In [10]:
EBB_final.groupby(['BranchName']).sum()
EBB_final.groupby('BranchName')[['IsChildren','IsTeens','IsAdults','IsYoungerAdults','IsSeniors']].sum()

,IsChildren,IsTeens,IsAdults,IsYoungerAdults,IsSeniors
BranchName,,,,,
Agincourt,40,21,39,3,25
Albert Campbell,53,11,59,4,21
Albion,60,26,19,1,0
Alderwood,20,0,2,0,1
Amesbury Park,67,3,32,2,20
...,...,...,...,...,...
Woodside Square,20,14,27,5,21
Woodview Park,118,4,35,0,65
Wychwood,34,1,9,1,8


2. Graph for each brench on the events thy offer that concerns 
 public speaking workshops, senior digital 
literacy, volunteer fairs, or youth leadership programming

In [11]:
unique_categories = merged_data['EventTypes'].str.split(',').explode().str.strip().unique()
exp = merged_data.groupby(['BranchName', 'EventTypes'])
# Sort them alphabetically to make it easier to read

print(unique_categories)
exp.head(5)

<StringArray>
[                          'Culture',              'Arts & Entertainment',
                  'Crafts & Hobbies',                    'Games & Sports',
              'Personal Development',     'Reading Programs & Storytimes',
             'Science & Engineering',                 'Health & Wellness',
       'Book Clubs & Writers Groups',      'Civic & Community Engagement',
                 'Coding & Robotics',              'Conversation Circles',
                  'Personal Finance', 'Small Business & Entrepreneurship',
               'Career & Job Search',           'Author Talks & Lectures',
 'Computer Basics & Office Software',                       '3D Printing',
                    'Audio & Visual',                    'Digital Safety',
            'Library Visits & Tours',          'Nature & the Environment',
           'Artificial Intelligence',               'History & Genealogy',
                 'Language Learning',                          'Exhibits',
           

,BranchCode,PhysicalBranch,BranchName,KidsStop,CLC,DIH,TeenCouncil,YouthHub,Workstations,NBHDName,...,IsRecurring,IsFull,Status,RegistrationIsFull,LastUpdatedOn,IsChildren,IsTeens,IsSeniors,IsAdults,IsYoungerAdults
0,CL,1.0,North York Central Library,1.0,1.0,1.0,1.0,1.0,98.0,Willowdale West,...,False,False,ACTIVE,False,2026-02-26T12:22:14,False,False,True,True,False
1,CL,1.0,North York Central Library,1.0,1.0,1.0,1.0,1.0,98.0,Willowdale West,...,False,False,ACTIVE,False,2026-02-26T12:22:14,False,False,True,True,False
2,SWS,1.0,S. Walter Stewart,1.0,1.0,0.0,0.0,1.0,39.0,Danforth Village - East York,...,True,False,ACTIVE,False,2026-02-26T12:22:14,False,True,False,False,False
3,SWS,1.0,S. Walter Stewart,1.0,1.0,0.0,0.0,1.0,39.0,Danforth Village - East York,...,True,False,ACTIVE,False,2026-02-26T12:22:14,False,False,True,False,False
4,PL,1.0,Parliament Street,0.0,0.0,0.0,0.0,1.0,12.0,Moss Park,...,True,False,ACTIVE,False,2026-02-26T12:22:14,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8498,BF,1.0,Barbara Frum,0.0,1.0,0.0,1.0,1.0,30.0,Englemount-Lawrence,...,True,False,ACTIVE,NaN,2026-02-26T12:22:14,False,True,False,False,False
8504,FV,1.0,Fairview,1.0,1.0,0.0,1.0,1.0,55.0,Don Valley Village,...,False,False,ACTIVE,False,2026-02-26T12:22:14,True,True,True,True,True
8507,TH,1.0,Thorncliffe,1.0,0.0,0.0,0.0,1.0,19.0,Thorncliffe Park,...,False,False,ACTIVE,False,2026-02-26T12:22:14,False,False,True,True,True
8509,AB,1.0,Albion,1.0,1.0,1.0,1.0,1.0,38.0,Mount Olive-Silverstone-Jamestown,...,False,False,ACTIVE,False,2026-02-26T12:22:14,False,False,False,True,False


In [ ]:
# Preparing Graph2: Step 1 - Taking only relvent information

audience_cols = ['IsChildren','IsTeens','IsAdults','IsYoungerAdults','IsSeniors']

Events_by_type = merged_data[[
    'EventTypes', 
    'BranchName', 
    'IsChildren', 
    'IsTeens', 
    'IsAdults', 
    'IsYoungerAdults', 
    'IsSeniors'
]].copy()


# Now each row = one event x one type
# Group and sum — this counts how many events of each type served each audience
final_counts = Events_by_type.groupby(
    ['BranchName', 'EventTypes']
)[audience_cols].sum().astype(int)

final_counts.groupby('BranchName')[audience_cols].sum()
final_counts.groupby('EventTypes')[audience_cols].sum()
# branch_totals = final_counts.groupby('BranchName')[audience_cols].sum()
# print(branch_totals)
final_counts.head()


                                                               IsChildren  \
BranchName EventTypes                                                       
Agincourt  3D Printing                                                  0   
           Artificial Intelligence                                      0   
           Audio & Visual                                               1   
           Audio & Visual, Coding & Robotics, Computer Bas...           1   
           Audio & Visual, Computer Basics & Office Software            0   
...                                                                   ...   
Yorkville  Exhibits                                                     0   
           Health & Wellness                                            0   
           Nature & the Environment                                     1   
           Reading Programs & Storytimes                                8   
           Science & Engineering                                        1   

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# ── U+ alignment keywords ──────────────────────────────────────────────
UPLUS_KEYWORDS = [
    'personal development', 'career', 'civic', 'leadership',
    'digital', 'technology', 'computer', 'health', 'community',
    'volunteer', 'small business', 'conversation'
]

def is_uplus(event_type):
    lower = event_type.lower()
    return any(k in lower for k in UPLUS_KEYWORDS)

# ── Audience colours ───────────────────────────────────────────────────
AUD_COLORS = {
    'IsChildren':      '#3b82f6',
    'IsTeens':         '#22c55e',
    'IsAdults':        '#a855f7',
    'IsYoungerAdults': '#f97316',
    'IsSeniors':       '#eab308',
}
AUD_LABELS = {
    'IsChildren': 'Children', 'IsTeens': 'Teens',
    'IsAdults': 'Adults', 'IsYoungerAdults': 'Younger Adults', 'IsSeniors': 'Seniors'
}
audience_cols = list(AUD_LABELS.keys())

branch_list = sorted(final_counts.index.get_level_values('BranchName').unique().tolist())

# ── Widgets ────────────────────────────────────────────────────────────
branch_dropdown = widgets.Dropdown(
    options=branch_list,
    description='Branch:',
    layout=widgets.Layout(width='320px')
)

search_box = widgets.Text(
    placeholder='Search branch name…',
    description='Search:',
    layout=widgets.Layout(width='280px')
)

uplus_toggle = widgets.ToggleButton(
    value=False,
    description='Show U+ only',
    button_style='info',
    icon='filter',
    layout=widgets.Layout(width='150px', height='36px')
)

aud_toggles = {
    col: widgets.ToggleButton(
        value=True,
        description=AUD_LABELS[col],
        button_style='',
        layout=widgets.Layout(width='130px', height='32px')
    )
    for col in audience_cols
}

output = widgets.Output()

# ── Render function ────────────────────────────────────────────────────
def render(branch):
    with output:
        output.clear_output(wait=True)

        active_cols = [c for c, t in aud_toggles.items() if t.value]
        if not active_cols:
            print("Select at least one audience.")
            return

        try:
            df = final_counts.loc[branch].copy()
        except KeyError:
            print(f"No data for {branch}")
            return

        df = df.reset_index()
        df['uplus'] = df['EventTypes'].apply(is_uplus)
        df['total'] = df[active_cols].sum(axis=1)
        df = df[df['total'] > 0]

        # Apply U+ filter if toggled on
        if uplus_toggle.value:
            df = df[df['uplus'] == True]

        df = df.sort_values('total', ascending=False)

        # ── Metric summary ─────────────────────────────────────────────
        # Always show totals for the full branch (not filtered)
        full_df = final_counts.loc[branch]
        totals = {c: int(full_df[c].sum()) for c in audience_cols}

        uplus_note = " <span style='background:#dbeafe;color:#1e40af;font-size:11px;padding:2px 8px;border-radius:10px;margin-left:8px;'>Showing U+ aligned only</span>" if uplus_toggle.value else ""

        metric_html = "<div style='display:flex;gap:16px;margin-bottom:20px;flex-wrap:wrap;'>"
        for col in audience_cols:
            color = AUD_COLORS[col]
            label = AUD_LABELS[col]
            metric_html += f"""
            <div style='background:#f8fafc;border-radius:8px;padding:12px 18px;
                        border-left:4px solid {color};min-width:110px;text-align:center;'>
              <div style='font-size:24px;font-weight:700;color:{color};'>{totals[col]}</div>
              <div style='font-size:11px;color:#6b7280;margin-top:2px;'>{label}</div>
              <div style='font-size:10px;color:#9ca3af;margin-top:1px;'>events</div>
            </div>"""
        metric_html += "</div>"

        display(HTML(
            f"<h3 style='margin-bottom:6px;font-family:sans-serif;'>{branch}{uplus_note}</h3>"
            f"<p style='font-size:12px;color:#9ca3af;margin-bottom:14px;font-family:sans-serif;'>Total events per audience group (all program types)</p>"
            + metric_html
        ))

        # ── Event type table ───────────────────────────────────────────
        if df.empty:
            display(HTML("<p style='font-family:sans-serif;color:#9ca3af;'>No U+ aligned events found at this branch.</p>"))
            return

        event_count = len(df)
        filter_label = "U+ aligned event types" if uplus_toggle.value else "all event types"
        display(HTML(
            f"<p style='font-size:12px;color:#6b7280;font-family:sans-serif;margin-bottom:8px;'>"
            f"Showing <strong>{event_count}</strong> {filter_label}</p>"
        ))

        table_rows = ""
        for _, row in df.iterrows():
            et = row['EventTypes']
            up = row['uplus']
            bg = "#eff6ff" if up else "white"
            tag = " <span style='background:#dbeafe;color:#1e40af;font-size:10px;padding:2px 7px;border-radius:10px;font-weight:600;'>U+</span>" if up else ""
            cells = ""
            for c in audience_cols:
               color = AUD_COLORS[c] if c in active_cols else "#d1d5db"   # ← no f-string escaping needed
               cells += (
                    f"<td style='text-align:right;padding:6px 10px;color:{color};'>"
                    f"{int(row[c])} <span style='font-size:10px;color:#9ca3af;'>ev</span></td>"
        )
    
            table_rows += (
                f"<tr style='background:{bg};border-bottom:1px solid #f1f5f9;'>"
                f"<td style='padding:6px 10px;font-family:sans-serif;font-size:13px;'>{et}{tag}</td>"
                f"{cells}</tr>"
    )
        header_cells = "".join(
            f"<th style='text-align:right;padding:8px 10px;color:#6b7280;font-size:11px;"
            f"text-transform:uppercase;letter-spacing:0.05em;'>{AUD_LABELS[c]}<br>"
            f"<span style='font-weight:400;color:#9ca3af;font-size:10px;'>(events)</span></th>"
            for c in audience_cols
        )
        table_html = f"""
        <div style='overflow-x:auto;margin-bottom:20px;font-family:sans-serif;'>
        <table style='border-collapse:collapse;width:100%;font-size:13px;'>
          <thead style='background:#f8fafc;border-bottom:2px solid #e2e8f0;'>
            <tr>
              <th style='text-align:left;padding:8px 10px;color:#6b7280;font-size:11px;
                         text-transform:uppercase;letter-spacing:0.05em;'>
                Event type
                <span style='font-weight:400;font-size:10px;margin-left:6px;'>★ = U+ aligned</span>
              </th>
              {header_cells}
            </tr>
          </thead>
          <tbody>{table_rows}</tbody>
        </table></div>"""
        display(HTML(table_html))

        # ── Bar chart ──────────────────────────────────────────────────
        top10 = df.head(10)
        max_val = top10['total'].max() if len(top10) else 1

        bars = ""
        for _, row in top10.iterrows():
            et = row['EventTypes']
            label = et[:28] + "…" if len(et) > 28 else et
            pct = int(row['total'] / max_val * 100)
            bars += f"""
            <div style='display:flex;align-items:center;gap:10px;margin-bottom:8px;'>
              <div style='width:180px;text-align:right;font-size:12px;color:#374151;
                          flex-shrink:0;' title='{et}'>{label}</div>
              <div style='flex:1;background:#f1f5f9;border-radius:4px;height:18px;overflow:hidden;'>
                <div style='width:{pct}%;height:100%;
                            background:{"#4f46e5" if uplus_toggle.value else "#a855f7"};
                            border-radius:4px;'></div>
              </div>
              <div style='font-size:12px;color:#6b7280;width:52px;'>
                {int(row["total"])} ev
              </div>
            </div>"""

        chart_title = "Top U+ aligned event types" if uplus_toggle.value else "Top event types by total events"
        display(HTML(f"""
        <div style='font-family:sans-serif;'>
          <p style='font-size:13px;font-weight:600;color:#374151;margin-bottom:12px;'>
            {chart_title}
          </p>
          {bars}
        </div>"""))

# ── Wire up interactions ───────────────────────────────────────────────
def on_branch_change(change):
    if change['name'] == 'value':
        render(branch_dropdown.value)

def on_search(change):
    q = search_box.value.lower()
    filtered = [b for b in branch_list if q in b.lower()]
    branch_dropdown.options = filtered if filtered else branch_list
    if filtered:
        render(filtered[0])

def on_toggle(change):
    render(branch_dropdown.value)

branch_dropdown.observe(on_branch_change)
search_box.observe(on_search, names='value')
uplus_toggle.observe(on_toggle, names='value')
for t in aud_toggles.values():
    t.observe(on_toggle, names='value')

# ── Layout & display ───────────────────────────────────────────────────
toggle_row  = widgets.HBox(list(aud_toggles.values()))
controls    = widgets.HBox([branch_dropdown, search_box, uplus_toggle])

display(controls, toggle_row, output)
render(branch_list[0])

                                                               IsChildren  \
BranchName EventTypes                                                       
Agincourt  3D Printing                                                  0   
           Artificial Intelligence                                      0   
           Audio & Visual                                               1   
           Audio & Visual, Coding & Robotics, Computer Bas...           1   
           Audio & Visual, Computer Basics & Office Software            0   
...                                                                   ...   
Yorkville  Exhibits                                                     0   
           Health & Wellness                                            0   
           Nature & the Environment                                     1   
           Reading Programs & Storytimes                                8   
           Science & Engineering                                        1   

Output()

In [ ]:
# ── Top 15 branches aligned with U+ ───────────────────────────────────
all_branches = final_counts.index.get_level_values('BranchName').unique().tolist()

rows = []
for branch in all_branches:
    df_b = final_counts.loc[branch].reset_index()
    df_b['uplus'] = df_b['EventTypes'].apply(is_uplus)
    uplus_df = df_b[df_b['uplus'] == True]
    
    total_uplus_events = int(uplus_df[audience_cols].sum().sum())  # total events across all audiences
    uplus_types        = uplus_df['EventTypes'].nunique()           # how many distinct program types
    senior_events      = int(uplus_df['IsSeniors'].sum())
    teen_events        = int(uplus_df['IsTeens'].sum())
    adult_events       = int(uplus_df['IsAdults'].sum())
    ya_events          = int(uplus_df['IsYoungerAdults'].sum())

    rows.append({
        'Branch':            branch,
        'U+ Events':         total_uplus_events,
        'U+ Program Types':  uplus_types,
        'Adults (ev)':       adult_events,
        'Younger Adults (ev)': ya_events,
        'Teens (ev)':        teen_events,
        'Seniors (ev)':      senior_events,
    })

top15 = (
    pd.DataFrame(rows)
    .sort_values('U+ Events', ascending=False)
    .head(15)
    .reset_index(drop=True)
)
top15.index += 1  # rank starts at 1

# ── Render table ───────────────────────────────────────────────────────
header_cols = ['Branch', 'U+ Events', 'U+ Program Types', 'Adults (ev)', 'Younger Adults (ev)', 'Teens (ev)', 'Seniors (ev)']

header_cells = "".join(
    f"<th style='padding:9px 12px;text-align:{'left' if c == 'Branch' else 'right'};"
    f"font-size:11px;color:#6b7280;text-transform:uppercase;letter-spacing:0.05em;"
    f"background:#f8fafc;border-bottom:2px solid #e2e8f0;'>{c}</th>"
    for c in header_cols
)

col_colors = {
    'U+ Events':           '#4f46e5',
    'U+ Program Types':    '#374151',
    'Adults (ev)':         AUD_COLORS['IsAdults'],
    'Younger Adults (ev)': AUD_COLORS['IsYoungerAdults'],
    'Teens (ev)':          AUD_COLORS['IsTeens'],
    'Seniors (ev)':        AUD_COLORS['IsSeniors'],
}

table_rows = ""
for rank, row in top15.iterrows():
    cells = f"<td style='padding:8px 12px;font-size:13px;font-weight:500;'>{row['Branch']}</td>"
    for c in header_cols[1:]:
        color = col_colors.get(c, '#374151')
        cells += (
            f"<td style='padding:8px 12px;text-align:right;font-size:13px;"
            f"font-weight:600;color:{color};'>{row[c]}</td>"
        )
    bg = "#f5f3ff" if rank == 1 else ("white" if rank % 2 == 0 else "#fafafa")
    table_rows += (
        f"<tr style='background:{bg};border-bottom:1px solid #f1f5f9;'>"
        f"<td style='padding:8px 12px;font-size:12px;color:#9ca3af;width:32px;'>{rank}</td>"
        + cells +
        "</tr>"
    )

display(HTML(f"""
<div style='font-family:sans-serif;margin-top:16px;'>
  <h3 style='margin-bottom:4px;'>Top 15 Branches — U+ Program Alignment</h3>
  <p style='font-size:12px;color:#9ca3af;margin-bottom:14px;'>
    Ranked by total events across U+ aligned program types. 
    U+ keywords: personal development, career, civic, leadership, digital, technology, 
    computer, health, community, volunteer, small business, conversation.
  </p>
  <div style='overflow-x:auto;'>
  <table style='border-collapse:collapse;width:100%;font-size:13px;'>
    <thead><tr><th style='padding:9px 12px;background:#f8fafc;border-bottom:2px solid #e2e8f0;
                           font-size:11px;color:#6b7280;'>#</th>
      {header_cells}
    </tr></thead>
    <tbody>{table_rows}</tbody>
  </table>
  </div>
</div>
"""))

#,Branch,U+ Events,U+ Program Types,Adults (ev),Younger Adults (ev),Teens (ev),Seniors (ev)
1,Sanderson,316,4,164,0,0,152
2,Toronto Reference Library,264,10,142,8,5,109
3,Malvern,164,7,156,0,0,8
4,Parliament Street,128,6,81,0,1,9
5,Cedarbrae,106,6,43,15,22,26
6,Northern District,99,7,74,4,5,16
7,Lillian H. Smith,99,6,91,2,1,4
8,Woodview Park,96,3,32,0,0,32
9,Parkdale,94,3,81,0,0,13
10,Eatonville,90,2,2,0,87,1


                                                               IsChildren  \
BranchName EventTypes                                                       
Agincourt  3D Printing                                                  0   
           Artificial Intelligence                                      0   
           Audio & Visual                                               1   
           Audio & Visual, Coding & Robotics, Computer Bas...           1   
           Audio & Visual, Computer Basics & Office Software            0   
...                                                                   ...   
Yorkville  Exhibits                                                     0   
           Health & Wellness                                            0   
           Nature & the Environment                                     1   
           Reading Programs & Storytimes                                8   
           Science & Engineering                                        1   

In [43]:
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. PREPARE THE PAGES
# Page A: The Interactive U+ Explorer (Reuse your existing widgets)
# Note: We wrap them in a VBox for layout control
page1 = widgets.VBox([
    widgets.HTML("<h3>🔍 Program Alignment Explorer</h3>"),
    widgets.HBox([branch_dropdown, search_box, uplus_toggle]),
    widgets.HBox(list(aud_toggles.values())),
    output  # This is the output widget your render function uses
])

# Page B: Top 15 Branches (Using a simple HTML widget for the table)
# Assuming 'top15' dataframe exists from your previous code
page2 = widgets.VBox([
    widgets.HTML("<h3>🏆 Top 15 Branches Leaderboard</h3>"),
    widgets.HTML(top15.to_html(classes='table table-hover'))
])

# Page C: Age Group Analysis
# We create a new output for the matplotlib graphs to keep things separate
age_output = widgets.Output()

def update_age_plot(change):
    with age_output:
        age_output.clear_output(wait=True)
        plot_branch(change['new']) # Using your original plot function

age_dropdown = widgets.Dropdown(options=EBB_final.index.tolist(), description="Branch:")
age_dropdown.observe(update_age_plot, names='value')

page3 = widgets.VBox([
    widgets.HTML("<h3>📊 Age Group Distribution</h3>"),
    age_dropdown,
    age_output
])

# 2. ASSEMBLE INTO A TAB WIDGET
dashboard = widgets.Tab()
dashboard.children = [page1, page2, page3]

# Set Titles
dashboard.set_title(0, 'U+ Explorer')
dashboard.set_title(1, 'Top Rankings')
dashboard.set_title(2, 'Age Analysis')

# 3. DISPLAY
display(HTML("<h2 style='color:#185FA5;'>TPL Analyst Dashboard</h2>"))
display(dashboard)

# Trigger initial renders so the tabs aren't empty on load
render(branch_dropdown.value)
update_age_plot({'new': age_dropdown.value})

In [17]:
# Create a single HTML string
html_content = f"""
<html>
<head>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f9; }}
        .card {{ background: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); margin-bottom: 20px; }}
        h1 {{ color: #185FA5; }}
        table {{ width: 100%; border-collapse: collapse; margin-top: 20px; }}
        th, td {{ padding: 12px; border-bottom: 1px solid #ddd; text-align: left; }}
        th {{ background-color: #185FA5; color: white; }}
    </style>
</head>
<body>
    <h1>TPL Final Analysis Dashboard</h1>
    
    <div class="card">
        <h2>Top 15 Branches — U+ Alignment</h2>
        {top15.to_html(index=False)}
    </div>

    <div class="card">
        <h2>Analyst Conclusion</h2>
        <ul>
            <li><strong>Expansion Opportunity:</strong> Focus on branches with high Senior engagement but low digital literacy programs.</li>
            <li><strong>Youth Leadership:</strong> Top performing branches for Teens should be used as models for lower-tier branches.</li>
        </ul>
    </div>
</body>
</html>
"""

# Save to file
with open("dashboard.html", "w") as f:
    f.write(html_content)

print("Dashboard exported to dashboard.html!")

Dashboard exported to dashboard.html!


In [ ]:
print(final_counts.reset_index().columns)

Index(['BranchName', 'EventTypes', 'IsChildren', 'IsTeens', 'IsAdults',
       'IsYoungerAdults', 'IsSeniors'],
      dtype='str')
                                                               IsChildren  \
BranchName EventTypes                                                       
Agincourt  3D Printing                                                  0   
           Artificial Intelligence                                      0   
           Audio & Visual                                               1   
           Audio & Visual, Coding & Robotics, Computer Bas...           1   
           Audio & Visual, Computer Basics & Office Software            0   
...                                                                   ...   
Yorkville  Exhibits                                                     0   
           Health & Wellness                                            0   
           Nature & the Environment                                     1   
           Reading Pro

In [54]:
import json

# 1. Prepare data for the HTML (converting your DataFrame to JSON for JavaScript)
branch_data_json = final_counts.reset_index().to_json(orient='records')
uplus_keywords_json = json.dumps(UPLUS_KEYWORDS)
top15_html = top15.to_html(index=False, classes="min-w-full text-sm text-left text-slate-600")

# 2. Plain Python string (NO 'f' prefix). JavaScript's ${} syntax is safe here!
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>TPL Analyst Dashboard</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <style>
        .metric-card { border-left: 4px solid; }
        table { width: 100%; border-collapse: collapse; }
        th { background: #f8fafc; padding: 12px; border-bottom: 2px solid #e2e8f0; font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #6b7280; }
        td { padding: 12px; border-bottom: 1px solid #f1f5f9; }
    </style>
</head>
<body class="bg-slate-50 p-8">
    <div class="max-w-6xl mx-auto">
        <h1 class="text-3xl font-bold text-blue-900 mb-8">Toronto Public Library Analysis</h1>

        <div class="bg-white rounded-xl shadow-sm p-6 mb-8">
            <h2 class="text-xl font-semibold mb-4 text-slate-700">🔍 Branch Explorer</h2>
            
            <div class="flex gap-4 mb-6">
                <select id="branchSelect" class="border p-2 rounded w-64 shadow-sm" onchange="updateDashboard()">
                </select>
                <div class="flex items-center gap-2">
                    <input type="checkbox" id="uplusFilter" onchange="updateDashboard()">
                    <label class="text-sm font-medium">Show U+ Only</label>
                </div>
            </div>

            <div id="metrics" class="flex flex-wrap gap-4 mb-8"></div>
            <div id="tableContainer" class="overflow-x-auto"></div>
        </div>

        <div class="bg-white rounded-xl shadow-sm p-6">
            <h2 class="text-xl font-semibold mb-4 text-slate-700">🏆 Top 15 Branches Leaderboard</h2>
            <div class="overflow-x-auto">
                __TOP15_LEADERBOARD_TABLE__
            </div>
        </div>
    </div>

    <script>
        const rawData = __BRANCH_DATA_JSON__;
        const branchList = [...new Set(rawData.map(d => d.BranchName))].sort();
        const uplusKeywords = __UPLUS_KEYWORDS_JSON__;
        
        const audColors = {
            'IsChildren': '#3b82f6',
            'IsTeens': '#22c55e',
            'IsAdults': '#a855f7',
            'IsYoungerAdults': '#f97316',
            'IsSeniors': '#eab308'
        };

        // Populate Dropdown
        const select = document.getElementById('branchSelect');
        branchList.forEach(b => {
            const opt = document.createElement('option');
            opt.value = b;
            opt.innerHTML = b;
            select.appendChild(opt);
        });

        function updateDashboard() {
            const branch = select.value;
            const showUPlus = document.getElementById('uplusFilter').checked;
            const branchData = rawData.filter(d => d.BranchName === branch);
            
            // Calculate Metrics
            const sums = {
                'IsChildren': 0, 'IsTeens': 0, 'IsAdults': 0, 
                'IsYoungerAdults': 0, 'IsSeniors': 0
            };
            branchData.forEach(d => {
                Object.keys(sums).forEach(k => sums[k] += d[k]);
            });

            // Render Metrics Cards
            const metricsDiv = document.getElementById('metrics');
            metricsDiv.innerHTML = Object.entries(sums).map(([label, val]) => `
                <div class="bg-slate-50 p-4 rounded-lg metric-card shadow-sm min-w-[140px]" style="border-color: ${audColors[label]}">
                    <div class="text-2xl font-bold" style="color: ${audColors[label]}">${val}</div>
                    <div class="text-xs text-slate-500 font-semibold uppercase mt-1">${label.replace('Is', '')}</div>
                </div>
            `).join('');

            // Filter main dataset loop
            let filteredData = branchData;
            if (showUPlus) {
                filteredData = branchData.filter(d => 
                    uplusKeywords.some(k => d.EventTypes.toLowerCase().includes(k))
                );
            }

            // Render Multi-Audience breakdown table cleanly using JavaScript templates
            const tableHtml = `
                <table>
                    <thead>
                        <tr>
                            <th class="text-left">Event Type</th>
                            <th class="text-right">Children</th>
                            <th class="text-right">Teens</th>
                            <th class="text-right">Adults</th>
                            <th class="text-right">Younger Adults</th>
                            <th class="text-right">Seniors</th>
                        </tr>
                    </thead>
                    <tbody>
                        ${filteredData.map(d => `
                            <tr>
                                <td class="text-sm font-medium text-slate-700">${d.EventTypes}</td>
                                <td class="text-right text-sm font-semibold" style="color: ${audColors.IsChildren}">${d.IsChildren} <span class="text-xs text-slate-400 font-normal">ev</span></td>
                                <td class="text-right text-sm font-semibold" style="color: ${audColors.IsTeens}">${d.IsTeens} <span class="text-xs text-slate-400 font-normal">ev</span></td>
                                <td class="text-right text-sm font-semibold" style="color: ${audColors.IsAdults}">${d.IsAdults} <span class="text-xs text-slate-400 font-normal">ev</span></td>
                                <td class="text-right text-sm font-semibold" style="color: ${audColors.IsYoungerAdults}">${d.IsYoungerAdults} <span class="text-xs text-slate-400 font-normal">ev</span></td>
                                <td class="text-right text-sm font-semibold" style="color: ${audColors.IsSeniors}">${d.IsSeniors} <span class="text-xs text-slate-400 font-normal">ev</span></td>
                            </tr>
                        `).join('')}
                    </tbody>
                </table>
            `;
            document.getElementById('tableContainer').innerHTML = tableHtml;
        }

        // Initial Load
        updateDashboard();
    </script>
</body>
</html>
"""

# 3. Swap placeholders with your real Python components safely
html_template = html_template.replace("__BRANCH_DATA_JSON__", branch_data_json)
html_template = html_template.replace("__UPLUS_KEYWORDS_JSON__", uplus_keywords_json)
html_template = html_template.replace("__TOP15_LEADERBOARD_TABLE__", top15_html)

# 4. Save to file
with open("DA08_Eden_ProjectFile.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Success! Open ' DA-08_Eden_ProjectFile.html' in your browser.")

Success! Open ' DA-08_Eden_ProjectFile.html' in your browser.
